# Part 4: Export Destinations

Section 4: Aduanas-derived port shares and COMTRADE cross-validation.

**Depends on:** Part 3

In [1]:
import os, shutil, re, pickle
from collections import Counter
import numpy as np
import pandas as pd
import openpyxl
import matplotlib.pyplot as plt
import seaborn as sns

BASE_DIR = "/Users/leoss/Desktop/GitHub/Capstone/Case studies/Chile"
DIR_PRELIM = os.path.join(BASE_DIR, "Preliminary")
COCHILCO_PATH = os.path.join(BASE_DIR, "data", "COCHILCO_Production_2005_2024.xlsx")

_cochilco_orig_candidates = [
    os.path.join(BASE_DIR, "data", "1771263160312_Anuario-de-Estadisticas-del-Cobre-y-otros-Minerales-2005-2024.xlsx"),
    os.path.join(BASE_DIR, "data", "Anuario-de-Estadisticas-del-Cobre-y-otros-Minerales-2005-2024.xlsx"),
    "/Users/leoss/Downloads/1771263160312_Anuario-de-Estadisticas-del-Cobre-y-otros-Minerales-2005-2024.xlsx",
    "/Users/leoss/Downloads/Anuario-de-Estadisticas-del-Cobre-y-otros-Minerales-2005-2024.xlsx",
]
COCHILCO_ORIG = next((p for p in _cochilco_orig_candidates if os.path.exists(p)), _cochilco_orig_candidates[0])

SALIDAS_PATH = os.path.join(BASE_DIR, "data", "salidas_2024_clean.csv")

# ── Load state from Part 3 ───────────────────────────────────────────
_state_path = os.path.join(DIR_PRELIM, "_pipeline_state_3.pkl")
with open(_state_path, "rb") as _f:
    _state = pickle.load(_f)

inv = _state["inv"]
links = _state["links"]
comm_col = _state.get("comm_col", "COMMODITY_LIST_STR")
idle_mines = _state.get("idle_mines", set())
COMPANY_TO_DEPOSIT = _state.get("COMPANY_TO_DEPOSIT", {})
CODELCO_EXTRA_SEARCH = _state.get("CODELCO_EXTRA_SEARCH", {})
SMELTERS = _state.get("SMELTERS", [])
PORTS = _state.get("PORTS", [])
SMELTER_NAME_MAP = _state.get("SMELTER_NAME_MAP", {})
DEDICATED_PORT = _state.get("DEDICATED_PORT", {})
CODELCO_CATHODE_ROUTING = _state.get("CODELCO_CATHODE_ROUTING", {})
MATCH_DISAMBIGUATION = _state.get("MATCH_DISAMBIGUATION", {})
IRON_MINE_NAMES = _state.get("IRON_MINE_NAMES", [])
ZINC_MINE_NAMES = _state.get("ZINC_MINE_NAMES", [])
edges = _state.get("edges", pd.DataFrame())
common_cols = _state.get("common_cols", [
    "FROM_NAME", "FROM_TYPE", "FROM_LAT", "FROM_LON",
    "TO_NAME", "TO_TYPE", "TO_LAT", "TO_LON",
    "EDGE_TYPE", "PRODUCT_FORM", "COMMODITIES", "DISTANCE_KM"])
smelter_inv_map = _state.get("smelter_inv_map", {})

# Maritime ports from state; extend with air freight ports for Gold/Silver
ports_df = pd.DataFrame(PORTS)
AIR_PORTS = [
    {"name": "Santiago (air)",    "region": "Metropolitana", "lat": -33.39, "lon": -70.79,
     "products": "gold, silver, air", "key_users": "Gold/Silver air freight"},
    {"name": "Antofagasta (air)", "region": "Antofagasta",   "lat": -23.44, "lon": -70.44,
     "products": "gold, silver, air", "key_users": "North air freight"},
    {"name": "Iquique (air)",     "region": "Tarapaca",      "lat": -20.54, "lon": -70.18,
     "products": "gold, silver, air", "key_users": "North air freight"},
    {"name": "Arica (air)",       "region": "Arica",         "lat": -18.35, "lon": -70.34,
     "products": "air", "key_users": "Far north air freight"},
    {"name": "Puerto Montt (air)","region": "Los Lagos",     "lat": -41.44, "lon": -73.09,
     "products": "air", "key_users": "South air freight"},
]
ports_df = pd.concat([ports_df, pd.DataFrame(AIR_PORTS)], ignore_index=True)

print(f"Loaded state from Part 2: {len(inv)} inv rows, {len(links)} link rows, {len(edges)} edges")

# ── Shared utility functions ──────────────────────────────────────────────

def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return R * 2 * np.arcsin(np.sqrt(a))

def parse_comm_list(val):
    if pd.isna(val): return []
    return [x.strip() for x in str(val).split(",") if x.strip()]

def add_commodity(row_idx, commodity, df, col):
    current = parse_comm_list(df.at[row_idx, col])
    if commodity not in current:
        current.append(commodity)
        df.at[row_idx, col] = ", ".join(current)
        return True
    return False

def nearest_port(lat, lon, product_type="concentrate"):
    best_dist, best_port = float("inf"), None
    for port in PORTS:
        if product_type == "cathode" and "cathode" not in port["products"].lower():
            continue
        if product_type == "concentrate" and "concentrate" not in port["products"].lower():
            continue
        dist = haversine_km(lat, lon, port["lat"], port["lon"])
        if dist < best_dist:
            best_dist, best_port = dist, port
    return best_port, best_dist

def section_header(title, width=65):
    print(f"\n{'=' * width}\n{title}\n{'=' * width}")

def search_inventory(inv_df, terms, require_mine=False):
    matched = set()
    name_lower = inv_df["FACILITY_NAME"].str.lower().str.strip()
    for term in terms:
        mask = name_lower.str.contains(term.lower(), na=False, regex=False)
        if require_mine:
            mask = mask & inv_df["FACILITY_TYPE"].str.contains("Mine", case=False, na=False)
        matched.update(inv_df[mask].index)
    return list(matched)


Loaded state from Part 2: 461 inv rows, 1109 link rows, 1165 edges


In [2]:

section_header("4. EXPORT DESTINATIONS")

wb = openpyxl.load_workbook(COCHILCO_ORIG, read_only=True, data_only=True)

def parse_destination_table(sheet_name, target_year=2024):
    ws = wb[sheet_name]
    rows = list(ws.iter_rows(values_only=True))
    yr_row, yc = None, None
    for i, row in enumerate(rows):
        if sum(1 for v in row if isinstance(v, (int, float)) and 2014 < v < 2025) >= 5:
            yr_row = i
            for j, v in enumerate(row):
                if isinstance(v, (int, float)) and int(v) == target_year:
                    yc = j
            break
    if yr_row is None or yc is None:
        print(f"  WARNING: Could not find {target_year} in {sheet_name}")
        return {}
    result = {}
    current_region = None
    for i in range(yr_row + 1, len(rows)):
        row = rows[i]
        label = str(row[0]).strip() if row[0] is not None else ""
        val = row[yc] if yc < len(row) else None
        if not label or label.startswith("(") or label.startswith("Fuente") or label.startswith("Source"):
            continue
        if any(r in label for r in ["EUROPA", "AMÉRICA", "ASIA", "OCEANÍA", "AFRICA"]):
            current_region = label.split("/")[0].strip()
            continue
        if label in ("TOTAL", "OTROS / Other"):
            continue
        if isinstance(val, (int, float)) and val > 0:
            parts = label.split("/")
            country = parts[-1].strip() if len(parts) > 1 else parts[0].strip()
            result[country] = {"value": val, "region": current_region or ""}
    return result

def parse_nonmetallic_column(sheet_name, col_idx, target_rows=(10, 65)):
    ws = wb[sheet_name]
    rows = list(ws.iter_rows(values_only=True))
    result, current_region = {}, None
    for i in range(target_rows[0], min(target_rows[1], len(rows))):
        row = rows[i]
        label = str(row[0]).strip() if row[0] is not None else ""
        val = row[col_idx] if col_idx < len(row) else None
        if not label: continue
        if any(r in label for r in ["EUROPA", "AMÉRICA", "ASIA"]):
            current_region = label.split("/")[0].strip()
            continue
        if label in ("TOTAL", "TOTAL MINERÍA"): continue
        if isinstance(val, (int, float)) and val > 0:
            parts = label.split("/")
            country = parts[-1].strip() if len(parts) > 1 else parts[0].strip()
            result[country] = {"value": val, "region": current_region or ""}
    return result

print("Parsing export destination tables...\n")
cu_refined = parse_destination_table("Tabla 18.2")
cu_blister = parse_destination_table("Tabla 19.2")
cu_concentrate = parse_destination_table("Tabla 20.2")
mo_concentrate = parse_destination_table("Tabla 23.2")
li_exports = parse_nonmetallic_column("Tabla 11", col_idx=2)
io_exports = parse_nonmetallic_column("Tabla 11", col_idx=7)

for name, data, unit in [
    ("Cu refined", cu_refined, "kMT"), ("Cu blister", cu_blister, "kMT"),
    ("Cu concentrate", cu_concentrate, "kMT"), ("Mo concentrate", mo_concentrate, "MT"),
    ("Lithium", li_exports, "$M FOB"), ("Iodine", io_exports, "$M FOB"),
]:
    total = sum(d["value"] for d in data.values())
    print(f"  {name:<20} {len(data):>3} destinations, total: {total:>10,.1f} {unit}")

wb.close()

# Country coordinates and aliases
COUNTRY_COORDS = {
    "China": {"lat": 31.23, "lon": 121.47}, "Japan": {"lat": 35.68, "lon": 139.69},
    "South Korea": {"lat": 37.57, "lon": 126.98}, "USA": {"lat": 29.76, "lon": -95.37},
    "Brazil": {"lat": -23.55, "lon": -46.63}, "India": {"lat": 19.08, "lon": 72.88},
    "Germany": {"lat": 53.55, "lon": 9.99}, "Spain": {"lat": 36.72, "lon": -4.42},
    "France": {"lat": 48.86, "lon": 2.35}, "Italy": {"lat": 45.46, "lon": 9.19},
    "Netherlands": {"lat": 51.92, "lon": 4.48}, "Belgium": {"lat": 51.26, "lon": 4.35},
    "Sweden": {"lat": 57.71, "lon": 11.97}, "Bulgaria": {"lat": 42.70, "lon": 23.32},
    "Finland": {"lat": 60.17, "lon": 24.94}, "Canada": {"lat": 49.28, "lon": -123.12},
    "Mexico": {"lat": 19.43, "lon": -99.13}, "Taiwan": {"lat": 25.03, "lon": 121.57},
    "Thailand": {"lat": 13.76, "lon": 100.50}, "Philippines": {"lat": 14.60, "lon": 120.98},
    "Malaysia": {"lat": 3.14, "lon": 101.69}, "Indonesia": {"lat": -6.21, "lon": 106.85},
    "Vietnam": {"lat": 10.82, "lon": 106.63}, "Peru": {"lat": -12.05, "lon": -77.04},
    "Colombia": {"lat": 4.71, "lon": -74.07}, "Argentina": {"lat": -34.60, "lon": -58.38},
    "Turkey": {"lat": 41.01, "lon": 28.98}, "United Kingdom": {"lat": 51.51, "lon": -0.13},
    "Switzerland": {"lat": 47.38, "lon": 8.54}, "Singapore": {"lat": 1.35, "lon": 103.82},
    "Greece": {"lat": 37.98, "lon": 23.73}, "Portugal": {"lat": 38.72, "lon": -9.14},
    "Panama": {"lat": 8.98, "lon": -79.52}, "Bahrain": {"lat": 26.07, "lon": 50.56},
    "UAE": {"lat": 25.20, "lon": 55.27}, "Hong Kong": {"lat": 22.32, "lon": 114.17},
    "Poland": {"lat": 52.23, "lon": 21.01}, "Norway": {"lat": 59.91, "lon": 10.75},
}

COUNTRY_ALIAS = {
    "Corea del Sur": "South Korea", "Estados Unidos": "USA",
    "Emiratos Árabes Unidos": "UAE", "United Arab Emirates": "UAE",
    "Baréin": "Bahrain", "Turquía": "Turkey", "Taiwán": "Taiwan",
    "Tailandia": "Thailand", "Filipinas": "Philippines",
    "Malasia": "Malaysia", "Singapur": "Singapore",
    "Panamá": "Panama", "Noruega": "Norway",
}

def normalize_country(name):
    return COUNTRY_ALIAS.get(name, name)

# ── 4A. ADUANAS-DERIVED PORT SHARES ─────────────────────────────────────

_salidas_candidates = [
    os.path.join(BASE_DIR, "data", "Salidas2024.csv"),
    os.path.join(BASE_DIR, "data", "Salidas2025.csv"),
]
SALIDAS_PATH = next((p for p in _salidas_candidates if os.path.exists(p)), _salidas_candidates[0])
CODIGOS_PATH = os.path.join(BASE_DIR, "data", "tablas_de_codigos.xlsx")

EXPORT_OP_CODES = {"200", "201", "202", "203", "204", "205", "206", "207",
                   "210", "211", "212", "213", "216"}

HS_MINERAL_MAP = {
    "2603": ("Copper", "concentrate"), "7402": ("Copper", "blister"),
    "7403": ("Copper", "cathode"), "2613": ("Molybdenum", "mo_concentrate"),
    "2601": ("Iron", "iron_ore"), "7108": ("Gold", "gold_refined"),
    "7106": ("Silver", "silver_refined"),
    "283620": ("Lithium", "lithium_compounds"), "28362010": ("Lithium", "lithium_compounds"),
    "28362020": ("Lithium", "lithium_compounds"), "28362030": ("Lithium", "lithium_compounds"),
    "283691": ("Lithium", "lithium_compounds"), "28369100": ("Lithium", "lithium_compounds"),
    "282520": ("Lithium", "lithium_compounds"), "28252000": ("Lithium", "lithium_compounds"),
    "284290": ("Lithium", "lithium_compounds"),
    "280120": ("Iodine", "iodine"), "2528": ("Boron", "borate"),
    "2602": ("Manganese", "mn_ore"), "2834": ("Nitrate", "nitrate"),
    "284170": ("Rhenium", "perrhenate"),
}

ADUANAS_PORT_ALIAS = {
    "CALETA COLOSO": "Coloso", "PUERTO ANGAMOS": "Angamos",
    "ANTOFAGASTA": "Antofagasta (ATI)", "CHAÑARAL / BARQUITO": "Barquito",
    "HUASCO / GUACOLDA": "Huasco", "GUAYACÁN": "Guayacán",
    "CAP. HUACHIPATO": "Huachipato",
    "AEROP. A.M. BENITEZ": "Santiago (air)", "AEROP. CERRO MORENO": "Antofagasta (air)",
    "AEROP. CHACALLUTA": "Arica (air)", "AEROP. DIEGO ARACENA": "Iquique (air)",
    "AEROP. EL TEPUAL": "Puerto Montt (air)", "AEROP. C.I. DEL CAMPO": "Santiago (air)",
    "CHACABUCO / PUERTO AYSÉN": "Puerto Aysén",
    "TERMINAL PETROLERO ENAP": "ENAP terminal", "OTROS PUERTOS CHILENOS": "Other",
}
ADUANAS_MANUAL_PORTS = {
    827: "Unknown (827)",
    821: "Iquique",     # ZOFRI free trade zone — not in tablas_de_codigos
    818: "Valparaíso",  # minor volume ($2.4M), best geographic approximation
}

PORT_PRODUCT_MAP_FALLBACK = {
    "concentrate": {"Coloso": 0.35, "Mejillones": 0.20, "Patache": 0.15,
                    "Barquito": 0.08, "Coquimbo": 0.12, "Angamos": 0.05, "Caldera": 0.05},
    "cathode": {"Angamos": 0.25, "Mejillones": 0.15, "Antofagasta (ATI)": 0.15,
                "Iquique": 0.15, "San Antonio": 0.10, "Ventanas": 0.10,
                "Barquito": 0.05, "Coquimbo": 0.05},
    "blister": {"Mejillones": 0.40, "Ventanas": 0.30, "Barquito": 0.20, "Antofagasta (ATI)": 0.10},
}

def classify_hs(code_str):
    code = str(code_str).replace(".", "").replace(" ", "").strip()
    code_trimmed = code.rstrip("0") or code
    for prefix_len in [8, 6, 4]:
        for c in [code[:prefix_len], code_trimmed[:prefix_len]]:
            if c in HS_MINERAL_MAP:
                return HS_MINERAL_MAP[c]
    return None

def parse_codigos_sheet(path, sheet_name, key_col_name, value_col_name):
    _xl = pd.read_excel(path, sheet_name=sheet_name, header=None)
    header_row, key_idx, val_idx = None, None, None
    for i in range(min(15, len(_xl))):
        for j, v in enumerate(_xl.iloc[i].values):
            if pd.notna(v) and str(v).strip().upper() == key_col_name.upper():
                key_idx, header_row = j, i
                break
        if header_row is not None: break
    if header_row is None: return {}
    for j, v in enumerate(_xl.iloc[header_row].values):
        if pd.notna(v) and value_col_name.upper() in str(v).strip().upper():
            val_idx = j; break
    if val_idx is None: return {}
    result = {}
    for ii in range(header_row + 1, len(_xl)):
        k, v = _xl.iloc[ii, key_idx], _xl.iloc[ii, val_idx]
        if pd.isna(k): continue
        try: result[int(float(k))] = str(v).strip()
        except (ValueError, TypeError): pass
    return result

# ── Load and process Aduanas data ────────────────────────────────────────

aduanas_loaded = False
aduanas_port_country = None
PORT_PRODUCT_MAP = {}

if os.path.exists(SALIDAS_PATH):
    section_header(f"4A. ADUANAS PORT SHARES ({os.path.basename(SALIDAS_PATH)})")

    with open(SALIDAS_PATH, "r", encoding="utf-8", errors="replace") as f:
        first_line = f.readline()
    _sep = ";" if first_line.count(";") > first_line.count(",") else ","
    print(f"  Detected delimiter: {'semicolon' if _sep == ';' else 'comma'}")

    sal = pd.read_csv(SALIDAS_PATH, sep=_sep, low_memory=False, dtype=str,
                       encoding="utf-8", on_bad_lines="skip")
    sal.columns = [c.strip().upper() for c in sal.columns]
    print(f"  Salidas loaded: {len(sal):,} rows x {len(sal.columns)} cols")

    # Filter to export operations
    op_col = next((c for c in sal.columns if "TIPO_OPERACION" in c.upper()), None)
    if op_col:
        before = len(sal)
        sal = sal[sal[op_col].astype(str).str.strip().isin(EXPORT_OP_CODES)]
        print(f"  Export filter: {before:,} -> {len(sal):,} rows")

    # Map key columns
    _col_map = {}
    for c in sal.columns:
        cl = c.lower()
        if "item_sa" in cl or "arancel" in cl: _col_map.setdefault("hs", c)
        if "puerto" in cl and "embarque" in cl: _col_map.setdefault("port", c)
        if "pais" in cl and ("destino" in cl or "origen" in cl): _col_map.setdefault("country", c)
        if "fob" in cl: _col_map.setdefault("fob", c)

    required = {"hs", "port", "country", "fob"}
    if required.issubset(_col_map.keys()):
        hs_col, port_col, country_col, fob_col = _col_map["hs"], _col_map["port"], _col_map["country"], _col_map["fob"]

        # Classify mineral rows
        sal["_hs_clean"] = sal[hs_col].astype(str).str.replace(".", "", regex=False).str.strip()
        sal["_mineral"] = sal["_hs_clean"].apply(classify_hs)
        mineral = sal[sal["_mineral"].notna()].copy()
        mineral["COMMODITY"] = mineral["_mineral"].apply(lambda x: x[0])
        mineral["PRODUCT_FORM"] = mineral["_mineral"].apply(lambda x: x[1])
        mineral["FOB_USD"] = pd.to_numeric(mineral[fob_col].str.replace(",", "."), errors="coerce").fillna(0)
        mineral["PORT_CODE"] = pd.to_numeric(mineral[port_col], errors="coerce")
        mineral["COUNTRY_CODE"] = pd.to_numeric(mineral[country_col], errors="coerce")

        print(f"  Mineral rows: {len(mineral):,} / {len(sal):,} ({len(mineral)/len(sal)*100:.1f}%)")
        print(f"  Total mineral FOB: ${mineral['FOB_USD'].sum():,.0f}")
        print(f"\n  By commodity:")
        for comm, grp in mineral.groupby("COMMODITY"):
            print(f"    {comm:<15} {len(grp):>6} rows  ${grp['FOB_USD'].sum():>15,.0f}")

        # Load lookups from codigos
        port_lookup, country_lookup, transport_lookup, region_lookup = {}, {}, {}, {}
        if os.path.exists(CODIGOS_PATH):
            country_lookup = parse_codigos_sheet(CODIGOS_PATH, "Países", "COD_PAIS", "NOMBRE_PAIS")
            port_lookup_raw = parse_codigos_sheet(CODIGOS_PATH, "Puertos", "COD_PUERTO", "NOMBRE_PUERTO")
            transport_lookup = parse_codigos_sheet(CODIGOS_PATH, "Vías de Transporte", "COD_VIA_TRANSPORTE", "NOMBRE_VIA_TRANSPORTE")
            region_lookup = parse_codigos_sheet(CODIGOS_PATH, "Regiones", "COD_REGION_ORIGEN", "NOMBRE_REGION")

            for code, raw_name in port_lookup_raw.items():
                upper = raw_name.upper().strip()
                port_lookup[code] = ADUANAS_PORT_ALIAS.get(upper, raw_name.title())
            for code, name in ADUANAS_MANUAL_PORTS.items():
                port_lookup.setdefault(code, name)

            print(f"  Codigos: {len(country_lookup)} countries, {len(port_lookup)} ports")

        # Map codes to names
        mineral["PORT_NAME"] = mineral["PORT_CODE"].apply(
    lambda c: port_lookup.get(int(c)) if pd.notna(c) else None)
        for pc in mineral[mineral["PORT_NAME"].isna()]["PORT_CODE"].dropna().unique():
            vol = mineral[mineral["PORT_CODE"] == pc]["FOB_USD"].sum()
            if vol > 1_000_000:
                print(f"  Warning: unmapped port code {int(pc)}, FOB ${vol:,.0f}")

        # Transport mode summary
        via_col = next((c for c in sal.columns if "VIA_TRANSPORTE" in c.upper()), None)
        if via_col and transport_lookup:
            mineral["TRANSPORT_MODE"] = pd.to_numeric(mineral[via_col], errors="coerce").map(transport_lookup)
            print(f"\n  Transport modes (mineral FOB):")
            for mode, fob in mineral.groupby("TRANSPORT_MODE")["FOB_USD"].sum().sort_values(ascending=False).items():
                print(f"    {mode:<30} ${fob:>15,.0f}  ({fob/mineral['FOB_USD'].sum()*100:.1f}%)")

        # Map countries
        ADUANAS_COUNTRY_ALIAS = {
            **COUNTRY_ALIAS,
            "China": "China", "Japón": "Japan", "Alemania": "Germany",
            "España": "Spain", "Francia": "France", "Italia": "Italy",
            "Países Bajos": "Netherlands", "Bélgica": "Belgium",
            "Suecia": "Sweden", "Bulgaria": "Bulgaria", "Finlandia": "Finland",
            "Canadá": "Canada", "México": "Mexico", "Perú": "Peru",
            "Argentina": "Argentina", "Brasil": "Brazil",
            "Reino Unido": "United Kingdom", "Suiza": "Switzerland",
            "Grecia": "Greece", "Portugal": "Portugal",
            "India": "India", "Indonesia": "Indonesia",
            "Hong Kong": "Hong Kong", "Polonia": "Poland",
        }
        mineral["COUNTRY_NAME"] = mineral["COUNTRY_CODE"].map(
            lambda c: country_lookup.get(int(c), f"Code_{int(c)}") if pd.notna(c) else None)
        mineral["COUNTRY_EN"] = mineral["COUNTRY_NAME"].map(
            lambda x: ADUANAS_COUNTRY_ALIAS.get(x, x) if pd.notna(x) else None)

        # Compute PORT_PRODUCT_MAP from actual data
        cu_mineral = mineral[mineral["COMMODITY"] == "Copper"]
        for product in ["concentrate", "cathode", "blister"]:
            prod_data = cu_mineral[(cu_mineral["PRODUCT_FORM"] == product) & cu_mineral["PORT_NAME"].notna()]
            total_fob = prod_data["FOB_USD"].sum()
            if total_fob > 0:
                shares = (prod_data.groupby("PORT_NAME")["FOB_USD"].sum() / total_fob)
                PORT_PRODUCT_MAP[product] = shares[shares >= 0.005].to_dict()

        # Non-copper commodity port shares
        for comm in mineral["COMMODITY"].unique():
            if comm == "Copper": continue
            comm_data = mineral[(mineral["COMMODITY"] == comm) & mineral["PORT_NAME"].notna()]
            total_fob = comm_data["FOB_USD"].sum()
            if total_fob > 0:
                shares = comm_data.groupby("PORT_NAME")["FOB_USD"].sum() / total_fob
                product_form = comm_data["PRODUCT_FORM"].mode().iloc[0] if len(comm_data) > 0 else "unknown"
                PORT_PRODUCT_MAP[f"{comm.lower()}_{product_form}"] = shares[shares >= 0.005].to_dict()

        print(f"\n  PORT_PRODUCT_MAP derived for: {list(PORT_PRODUCT_MAP.keys())}")
        for key, shares in PORT_PRODUCT_MAP.items():
            top3 = sorted(shares.items(), key=lambda x: -x[1])[:3]
            print(f"    {key:<30} {len(shares)} ports  (top: {', '.join(f'{p} {s*100:.1f}%' for p, s in top3)})")

        # Port x country x commodity aggregation
        aduanas_port_country = mineral[
            mineral["PORT_NAME"].notna() & mineral["COUNTRY_EN"].notna()
        ].groupby(["COMMODITY", "PRODUCT_FORM", "PORT_NAME", "COUNTRY_EN"])["FOB_USD"].sum().reset_index()
        aduanas_port_country = aduanas_port_country[aduanas_port_country["FOB_USD"] > 0]
        print(f"\n  Aduanas port-country flows: {len(aduanas_port_country)} "
              f"({aduanas_port_country['COMMODITY'].nunique()} commodities, "
              f"{aduanas_port_country['PORT_NAME'].nunique()} ports, "
              f"{aduanas_port_country['COUNTRY_EN'].nunique()} countries)")

        # Save port shares CSV
        shares_rows = [{"PRODUCT": p, "PORT": port, "FOB_SHARE": share}
                       for p in ["concentrate", "cathode", "blister"]
                       if p in PORT_PRODUCT_MAP
                       for port, share in PORT_PRODUCT_MAP[p].items()]
        if shares_rows:
            pd.DataFrame(shares_rows).to_csv(os.path.join(DIR_PRELIM, "Chile_Port_Shares_Aduanas.csv"), index=False)
            print(f"  Saved: Chile_Port_Shares_Aduanas.csv ({len(shares_rows)} rows)")

        aduanas_loaded = True
    else:
        print(f"  ERROR: Could not map required columns. Found: {_col_map}")
else:
    print(f"\n  Salidas CSV not found at {SALIDAS_PATH}, using fallback PORT_PRODUCT_MAP")

# ── 4B. COMTRADE CROSS-VALIDATION ────────────────────────────────────────
# Compare Aduanas Salidas FOB totals against UN Comtrade aggregate data
# to flag discrepancies in HS classification or coverage.

COMTRADE_PATH = os.path.join(BASE_DIR, "data", "chile_mineral_trade_combined.csv")

# HS prefix -> commodity label mapping for Comtrade aggregation
COMTRADE_HS_MAP = {
    "2603": "Copper", "7402": "Copper", "7403": "Copper",
    "2613": "Molybdenum", "2601": "Iron", "7108": "Gold", "7106": "Silver",
    "2836": "Lithium", "2825": "Lithium", "2842": "Lithium",
    "2801": "Iodine", "2528": "Boron", "2602": "Manganese",
    "2834": "Nitrate", "2841": "Rhenium",
}

if os.path.exists(COMTRADE_PATH) and aduanas_loaded:
    section_header("4B. COMTRADE vs SALIDAS CROSS-VALIDATION")

    comtrade = pd.read_csv(COMTRADE_PATH)
    ct_exp = comtrade[comtrade["flowDesc"] == "Export"].copy()
    ct_exp["hs4"] = ct_exp["cmdCode"].astype(str).str[:4]
    ct_exp["commodity"] = ct_exp["hs4"].map(COMTRADE_HS_MAP)
    ct_exp = ct_exp[ct_exp["commodity"].notna()]

    # Use latest overlapping year (Salidas is typically one calendar year)
    ct_latest = ct_exp["period"].max()
    ct_yr = ct_exp[ct_exp["period"] == ct_latest]
    print(f"  Comtrade year: {ct_latest}  |  Salidas source: {os.path.basename(SALIDAS_PATH)}")
    print(f"  Comtrade mineral export rows: {len(ct_yr)}")

    # Aggregate Comtrade by commodity
    ct_agg = ct_yr.groupby("commodity").agg(
        ct_fob=("primaryValue", "sum"),
        ct_wgt_kg=("netWgt", "sum"),
        ct_partners=("partnerISO", "nunique"),
    ).sort_values("ct_fob", ascending=False)

    # Aggregate Salidas by commodity (mineral df from Section 4A)
    sal_agg = mineral.groupby("COMMODITY").agg(
        sal_fob=("FOB_USD", "sum"),
    )

    # Merge and compare
    comp = ct_agg.join(sal_agg, how="outer").fillna(0)
    comp["fob_ratio"] = comp.apply(
        lambda r: r["sal_fob"] / r["ct_fob"] if r["ct_fob"] > 0 else np.nan, axis=1)
    comp["fob_diff_pct"] = (comp["sal_fob"] - comp["ct_fob"]) / comp["ct_fob"] * 100

    print(f"\n  {'Commodity':<15} {'Comtrade FOB':>18} {'Salidas FOB':>18} {'Ratio':>8} {'Diff%':>8}  {'Comtrade kg':>18}")
    print("  " + "-" * 95)
    for comm, row in comp.iterrows():
        ratio_str = f"{row['fob_ratio']:.2f}" if pd.notna(row['fob_ratio']) else "N/A"
        diff_str = f"{row['fob_diff_pct']:+.1f}%" if pd.notna(row['fob_diff_pct']) else "N/A"
        flag = ""
        if pd.notna(row['fob_ratio']) and (row['fob_ratio'] < 0.80 or row['fob_ratio'] > 1.20):
            flag = " <-- MISMATCH"
        print(f"  {comm:<15} ${row['ct_fob']:>15,.0f}  ${row['sal_fob']:>15,.0f}  {ratio_str:>8}  {diff_str:>8}{flag}")

    # Top country-level comparison for copper (largest commodity)
    print(f"\n  Copper: top destination comparison (FOB)")
    ct_cu = ct_yr[ct_yr["commodity"] == "Copper"]

    # Map Comtrade ISO3 to country names used in pipeline
    ISO3_TO_NAME = {
        "CHN": "China", "JPN": "Japan", "KOR": "South Korea", "USA": "USA",
        "BRA": "Brazil", "IND": "India", "DEU": "Germany", "ESP": "Spain",
        "FRA": "France", "ITA": "Italy", "NLD": "Netherlands", "BEL": "Belgium",
        "SWE": "Sweden", "BGR": "Bulgaria", "FIN": "Finland", "CAN": "Canada",
        "MEX": "Mexico", "TWN": "Taiwan", "THA": "Thailand", "PHL": "Philippines",
        "MYS": "Malaysia", "IDN": "Indonesia", "VNM": "Vietnam", "PER": "Peru",
        "COL": "Colombia", "ARG": "Argentina", "TUR": "Turkey", "GBR": "United Kingdom",
        "CHE": "Switzerland", "SGP": "Singapore", "GRC": "Greece", "PRT": "Portugal",
        "PAN": "Panama", "BHR": "Bahrain", "ARE": "UAE", "HKG": "Hong Kong",
        "POL": "Poland", "NOR": "Norway",
    }

    ct_cu_by_country = ct_cu.groupby("partnerISO")["primaryValue"].sum().sort_values(ascending=False).head(10)
    # Compare against Aduanas port-country flows (aggregated to country level)
    if aduanas_port_country is not None:
        sal_cu = aduanas_port_country[aduanas_port_country["COMMODITY"] == "Copper"]
        sal_cu_by_country = sal_cu.groupby("COUNTRY_EN")["FOB_USD"].sum()

        print(f"  {'Country':<20} {'Comtrade FOB':>18} {'Salidas FOB':>18} {'Ratio':>8}")
        print("  " + "-" * 70)
        for iso3, ct_val in ct_cu_by_country.items():
            country_name = ISO3_TO_NAME.get(iso3, iso3)
            sal_val = sal_cu_by_country.get(country_name, 0)
            ratio = sal_val / ct_val if ct_val > 0 else np.nan
            ratio_str = f"{ratio:.2f}" if pd.notna(ratio) else "N/A"
            print(f"  {country_name:<20} ${ct_val:>15,.0f}  ${sal_val:>15,.0f}  {ratio_str:>8}")

    # Save cross-validation output
    comp.to_csv(os.path.join(DIR_PRELIM, "Comtrade_vs_Salidas_Validation.csv"))
    print(f"\n  Saved: Comtrade_vs_Salidas_Validation.csv")

elif not os.path.exists(COMTRADE_PATH):
    print(f"\n  Comtrade file not found at {COMTRADE_PATH}, skipping cross-validation")

# Apply fallback for missing product types
for key, fallback in PORT_PRODUCT_MAP_FALLBACK.items():
    if key not in PORT_PRODUCT_MAP:
        PORT_PRODUCT_MAP[key] = fallback

# ── Build port-to-country edges ─────────────────────────────────────────

def build_export_edges(dest_data, commodity, product_form, unit, port_shares):
    result = []
    port_dict = {p["name"]: p for p in ports_df.to_dict("records")}
    for country_raw, info in dest_data.items():
        country = normalize_country(country_raw)
        coords = COUNTRY_COORDS.get(country)
        if not coords: continue
        for port_name, share in port_shares.items():
            port = port_dict.get(port_name)
            if not port: continue
            result.append({
                "FROM_NAME": port_name, "FROM_TYPE": "port",
                "FROM_LAT": port["lat"], "FROM_LON": port["lon"],
                "TO_NAME": country, "TO_TYPE": "country",
                "TO_LAT": coords["lat"], "TO_LON": coords["lon"],
                "EDGE_TYPE": "port_to_country", "PRODUCT_FORM": product_form,
                "COMMODITIES": commodity, "DISTANCE_KM": None,
                "EXPORT_VALUE": round(info["value"] * share, 2),
                "EXPORT_UNIT": unit, "DESTINATION_TOTAL": round(info["value"], 2),
            })
    return result

def build_aduanas_edges(apc_df, commodity, product_form):
    sub = apc_df[(apc_df["COMMODITY"] == commodity) & (apc_df["PRODUCT_FORM"] == product_form)]
    port_dict = {p["name"]: p for p in ports_df.to_dict("records")}
    result = []
    for _, row in sub.iterrows():
        coords = COUNTRY_COORDS.get(row["COUNTRY_EN"])
        port = port_dict.get(row["PORT_NAME"])
        if not coords or not port: continue
        result.append({
            "FROM_NAME": row["PORT_NAME"], "FROM_TYPE": "port",
            "FROM_LAT": port["lat"], "FROM_LON": port["lon"],
            "TO_NAME": row["COUNTRY_EN"], "TO_TYPE": "country",
            "TO_LAT": coords["lat"], "TO_LON": coords["lon"],
            "EDGE_TYPE": "port_to_country", "PRODUCT_FORM": product_form,
            "COMMODITIES": commodity, "DISTANCE_KM": None,
            "EXPORT_VALUE": round(row["FOB_USD"], 2),
            "EXPORT_UNIT": "$FOB", "DESTINATION_TOTAL": round(row["FOB_USD"], 2),
        })
    return result

# Unified export edge construction: prefer Aduanas direct, fallback to COCHILCO proportional
EXPORT_CONFIGS = [
    ("Copper", "concentrate", cu_concentrate, "kMT", "concentrate"),
    ("Copper", "cathode", cu_refined, "kMT", "cathode"),
    ("Copper", "blister", cu_blister, "kMT", "blister"),
    ("Molybdenum", "mo_concentrate", mo_concentrate, "MT", "molybdenum_mo_concentrate"),
    ("Lithium", "lithium_compounds", li_exports, "$M_FOB", "lithium_lithium_compounds"),
    ("Iodine", "iodine", io_exports, "$M_FOB", "iodine_iodine"),
]

FALLBACK_SHARES = {
    "molybdenum_mo_concentrate": {"Mejillones": 0.50, "Antofagasta (ATI)": 0.30, "Barquito": 0.20},
    "lithium_lithium_compounds": {"Antofagasta (ATI)": 0.50, "Mejillones": 0.30, "Iquique": 0.20},
    "iodine_iodine": {"Iquique": 0.40, "Patache": 0.30, "Antofagasta (ATI)": 0.20, "Mejillones": 0.10},
}

ADUANAS_ONLY_COMMODITIES = [
    ("Gold", "gold_refined"), ("Silver", "silver_refined"),
    ("Iron", "iron_ore"), ("Manganese", "mn_ore"),
    ("Boron", "borate"), ("Nitrate", "nitrate"), ("Rhenium", "perrhenate"),
]

export_edges = []

for commodity, pform, dest_data, unit, port_key in EXPORT_CONFIGS:
    if aduanas_loaded and aduanas_port_country is not None:
        aduanas_edges = build_aduanas_edges(aduanas_port_country, commodity, pform)
        if aduanas_edges:
            export_edges.extend(aduanas_edges)
            continue
    # Fallback to proportional
    shares = PORT_PRODUCT_MAP.get(port_key, FALLBACK_SHARES.get(port_key, {}))
    export_edges.extend(build_export_edges(dest_data, commodity, pform, unit, shares))

# Non-COCHILCO commodities from Aduanas
if aduanas_loaded and aduanas_port_country is not None:
    for comm, pform in ADUANAS_ONLY_COMMODITIES:
        new_edges = build_aduanas_edges(aduanas_port_country, comm, pform)
        if new_edges:
            export_edges.extend(new_edges)
            total_fob = sum(e["EXPORT_VALUE"] for e in new_edges)
            print(f"  {comm:<15} {len(new_edges):>4} edges (${total_fob:,.0f} FOB)")

export_df = pd.DataFrame(export_edges)
print(f"\nExport edges: {len(export_df)}")
for commodity in export_df["COMMODITIES"].unique():
    sub = export_df[export_df["COMMODITIES"] == commodity]
    print(f"  {commodity:<15} {len(sub):>5} edges ({sub['TO_NAME'].nunique()} countries, {sub['FROM_NAME'].nunique()} ports)")

# Append to unified edge table
edges = edges[edges["EDGE_TYPE"] != "port_to_country"]
export_aligned = export_df[common_cols].copy()
for col in common_cols:
    if col not in export_aligned.columns:
        export_aligned[col] = ""
# Drop all-NA columns before concat to avoid FutureWarning on dtype inference
edges = edges.dropna(axis=1, how="all")
export_aligned = export_aligned.dropna(axis=1, how="all")
edges = pd.concat([edges, export_aligned], ignore_index=True)

print(f"\nUnified edges: {len(edges)}")
for et, count in edges["EDGE_TYPE"].value_counts().items():
    print(f"  {et:<25} {count:>5}")



# ── Save state after Part 3 ─────────────────────────────────────────
_prev_path = os.path.join(DIR_PRELIM, "_pipeline_state_3.pkl")
with open(_prev_path, "rb") as _f:
    _save_state = pickle.load(_f)
_save_state["inv"] = inv
_save_state["links"] = links
_save_state["edges"] = edges
_save_state["common_cols"] = common_cols
_save_state["export_df"] = export_df
_save_state["PORT_PRODUCT_MAP"] = PORT_PRODUCT_MAP
_out_path = os.path.join(DIR_PRELIM, "_pipeline_state_4.pkl")
with open(_out_path, "wb") as _f:
    pickle.dump(_save_state, _f)
print(f"State saved to {_out_path}")



4. EXPORT DESTINATIONS
Parsing export destination tables...

  Cu refined            26 destinations, total:    1,892.3 kMT
  Cu blister            14 destinations, total:      234.3 kMT
  Cu concentrate        24 destinations, total:    3,725.8 kMT
  Mo concentrate         9 destinations, total:   11,165.9 MT
  Lithium               10 destinations, total:    2,626.6 $M FOB
  Iodine                24 destinations, total:    1,449.7 $M FOB

4A. ADUANAS PORT SHARES (Salidas2025.csv)
  Detected delimiter: semicolon
  Salidas loaded: 328,443 rows x 19 cols
  Export filter: 328,443 -> 304,189 rows
  Mineral rows: 3,773 / 304,189 (1.2%)
  Total mineral FOB: $64,725,242,361

  By commodity:
    Boron               27 rows  $      3,572,284
    Copper            1591 rows  $ 53,226,061,073
    Gold               144 rows  $  3,405,134,725
    Iodine             383 rows  $  1,544,531,268
    Iron                70 rows  $  1,305,367,938
    Lithium            469 rows  $  2,171,282,233
    M